# 🧪 Sesión 08 · Respuestas Semánticas y Excepciones HTTP Personalizadas

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué debe ver el cliente cuando algo falla?

---

## 🖥️ Así se verá el resultado final

```json
GET /api/libros/99
HTTP/1.1 404 Not Found
{"error":{"codigo":"NO_ENCONTRADO","mensaje":"El libro 99 no existe"}}

POST /api/libros  {"titulo": ""}
HTTP/1.1 422 Unprocessable Entity
{"error":{"codigo":"VALIDACION","mensaje":"Revisa los datos enviados"}}

Nunca:
HTTP/1.1 500 Internal Server Error
{"detail":"Traceback (most recent call last)..."}
```

_El cliente necesita saber qué corregir; tú necesitas que no se filtre la implementación._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Lanzar excepciones de negocio con códigos HTTP semánticos.
- Centralizar el formato de error en un manejador global.
- Evitar respuestas 500 y fugas de detalles internos.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Un libro existente responde 200 → HTTP 200
- [ ] Un libro inexistente responde 404 con sobre de error → HTTP 404
- [ ] Los datos inválidos responden 422 con sobre de error → HTTP 422
- [ ] Un libro válido sigue respondiendo 201 → HTTP 201

## 🧩 El reto

Implementa un manejo de errores uniforme: una excepción de negocio para libros inexistentes y un manejador para los errores de validación.

**Requisitos del contrato**

- Define `LibroNoEncontrado` y regístralo con un handler que responda 404.
- El cuerpo del 404 es {"error": {"codigo": "NO_ENCONTRADO", "mensaje": ...}}.
- Los errores de validación responden 422 con el mismo sobre y codigo VALIDACION.
- Un título vacío debe rechazarse con 422.
- La API inicia con el libro 1 (Rayuela) para que el caso feliz exista.
- GET /api/libros/1 responde 200; GET /api/libros/99 responde 404.
- Ningún caso debe producir 500.

> **Pista conceptual:** FastAPI permite registrar handlers con `@app.exception_handler(...)`. El formato de error es parte del contrato, igual que el formato de éxito.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion08.md` con el formato de error elegido.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [4]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Biblioteca API")

libros = {
    1: {
        "id": 1,
        "titulo": "Rayuela",
        "autor": "Julio Cortázar"
    }
}

class Libro(BaseModel):
    titulo: str
    autor: str

@app.get("/api/libros/{id}")
def obtener_libro(id: int):
    if id in libros:
        return libros[id]
    else:
        raise HTTPException(
            status_code=404,
            detail="El libro no existe"
        )

@app.post("/api/libros", status_code=201)
def agregar_libro(libro: Libro):
    nuevo_id = len(libros) + 1

    libros[nuevo_id] = {
        "id": nuevo_id,
        "titulo": libro.titulo,
        "autor": libro.autor
    }

    return libros[nuevo_id]


Writing app.py


In [8]:
%%writefile app.py

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel, Field, ConfigDict

app = FastAPI(title="Biblioteca API")

libros = {
    1: {
        "id": 1,
        "titulo": "Rayuela",
        "autor": "Julio Cortázar"
    }
}

# Modelo de datos
class Libro(BaseModel):
    model_config = ConfigDict(extra="allow")

    titulo: str = Field(min_length=1)

# Excepción personalizada
class LibroNoEncontrado(Exception):
    def __init__(self, libro_id: int):
        self.libro_id = libro_id

# Error 404
@app.exception_handler(LibroNoEncontrado)
async def manejar_libro_no_encontrado(request: Request, exc: LibroNoEncontrado):
    return JSONResponse(
        status_code=404,
        content={
            "error": {
                "codigo": "NO_ENCONTRADO",
                "mensaje": f"El libro {exc.libro_id} no existe"
            }
        }
    )

# Error 422
@app.exception_handler(RequestValidationError)
async def manejar_validacion(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "error": {
                "codigo": "VALIDACION",
                "mensaje": "Revisa los datos enviados"
            }
        }
    )

# Consultar libros
@app.get("/api/libros/{id}")
def obtener_libro(id: int):
    if id not in libros:
        raise LibroNoEncontrado(id)

    return libros[id]

# Registrar libros
@app.post("/api/libros", status_code=201)
def agregar_libro(libro: Libro):
    nuevo_id = max(libros.keys()) + 1

    nuevo_libro = {
        "id": nuevo_id,
        **libro.model_dump()
    }

    libros[nuevo_id] = nuevo_libro

    return nuevo_libro

Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Estoy trabajando con Python y FastAPI en Google Colab. Utilizo Pydantic para validar los datos y probar las respuestas de la API. |
| **Rol** | Quiero que la IA actúe como un desarrollador web que me ayude a crear un manejo de errores para mi API. |
| **Especificación** | Crear una ruta para consultar libros y otra para registrar libros. Si un libro no existe debe responder 404. Si los datos son incorrectos debe responder 422. Si el registro es correcto debe responder 201 y si el libro existe debe responder 200. |
| **Output** | Necesito el código completo de app.py, con excepciones personalizadas y mensajes de error en formato JSON, listo para ejecutar en Google Colab. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [9]:
from verificar import probar_http

casos = [{'descripcion': 'Un libro existente responde 200',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1}},
 {'descripcion': 'Un libro inexistente responde 404 con sobre de error',
  'metodo': 'GET',
  'ruta': '/api/libros/99',
  'estado': 404,
  'json_contiene': {'error': {'codigo': 'NO_ENCONTRADO'}}},
 {'descripcion': 'Los datos inválidos responden 422 con sobre de error',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': ''},
  'estado': 422,
  'json_contiene': {'error': {'codigo': 'VALIDACION'}}},
 {'descripcion': 'Un libro válido sigue respondiendo 201',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Un libro existente responde 200
✅ 02. Un libro inexistente responde 404 con sobre de error
✅ 03. Los datos inválidos responden 422 con sobre de error
✅ 04. Un libro válido sigue respondiendo 201
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion08.md` a `auditorias/auditoria_sesion08.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?

   Me costó lograr que el registro de un libro respondiera con HTTP 201, porque al principio los datos eran rechazados por la validación y se recibía un error 422.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?

   Decidí utilizar manejadores globales para controlar los errores y mantener un formato uniforme en las respuestas de la API.

   - ¿Qué error detectó la verificación que tú no habías previsto?

   Detectó que el registro de un libro válido respondía con 422 en lugar de 201, debido a que el modelo de datos no coincidía con los datos enviados por la prueba.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
